In [1]:
!pip install transformer_lens torch numpy scikit-learn -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.9 MB/s eta 0:00:00


In [2]:
"""
m11_das_lite_pilot_v3.py
========================
Memory-optimized Milestone 11 DAS-Lite Pilot.

Fixes:
  1. Base model loaded in bfloat16 to reduce baseline VRAM.
  2. Batched tail forward pass: runs all K probe perturbations in a single batch (B=K).
  3. Prefix cache stored on CPU to prevent VRAM saturation.
"""

import torch
import torch.nn as nn
import numpy as np
import json
import gc
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. HARNESS (bfloat16 + CPU-cached prefix)
# ==============================================================================

class DASLiteHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.d_model = self.model.cfg.d_model
        self.inject_layer = 21
        self.tail_start_layer = 22

    @torch.no_grad()
    def get_prefix_cache(self, tokens: torch.Tensor) -> Tuple[torch.Tensor, int, int, float]:
        """Runs blocks 0..21 without gradients and caches residual stream on CPU."""
        _, cache = self.model.run_with_cache(
            tokens, stop_at_layer=self.tail_start_layer,
            names_filter=f"blocks.{self.inject_layer}.hook_resid_post"
        )
        resid = cache[f"blocks.{self.inject_layer}.hook_resid_post"].cpu()
        base_logits = self.model(resid.to(self.device), start_at_layer=self.tail_start_layer)[0, -1, :]
        top_ids = torch.topk(base_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((base_logits[t1] - base_logits[t2]).item())
        return resid, t1, t2, base_margin

    def batched_probe_response(self, resid: torch.Tensor, V_normed: torch.Tensor, scale: float,
                               t1: int, t2: int, base_margin: float) -> torch.Tensor:
        """Runs all K directions simultaneously in a single forward pass of batch size K."""
        K = V_normed.shape[0]
        # Expand resid from (1, seq, d_model) to (K, seq, d_model)
        resid_k = resid.to(self.device).repeat(K, 1, 1).to(torch.bfloat16)
        resid_k[:, -1, :] = resid_k[:, -1, :] + scale * V_normed.to(torch.bfloat16)

        logits = self.model(resid_k, start_at_layer=self.tail_start_layer)[:, -1, :]
        deltas = (logits[:, t1] - logits[:, t2]).to(torch.float32) - base_margin
        return deltas

    @torch.no_grad()
    def ground_truth_delta(self, tokens: torch.Tensor, t1: int, t2: int, base_margin: float,
                           layer: int, is_mlp: bool, head: int = 0, scale: float = 0.0) -> float:
        if is_mlp:
            def hook_fn(act, hook): act[:, -1, :] *= scale; return act
            hook_name = f"blocks.{layer}.hook_mlp_out"
        else:
            def hook_fn(act, hook): act[:, -1, head, :] *= scale; return act
            hook_name = f"blocks.{layer}.attn.hook_z"
        logits = self.model.run_with_hooks(tokens, fwd_hooks=[(hook_name, hook_fn)])[0, -1, :]
        return float((logits[t1] - logits[t2]).item()) - base_margin


# ==============================================================================
# 2. TARGET DEFINITIONS
# ==============================================================================

META_TRAIN_TARGETS = [
    ("L22H05_Ablate", 22, False, 5, 0.0),
    ("L22H00_Ablate", 22, False, 0, 0.0),
    ("L21MLP_Damp",   21, True,  0, 0.5),
    ("L18MLP_Damp",   18, True,  0, 0.5),
]

HELD_OUT_TARGETS = [
    ("L21H02_Ablate", 21, False, 2, 0.0),
    ("L14H04_Ablate", 14, False, 4, 0.0),
    ("L10H00_Ablate", 10, False, 0, 0.0),
    ("L23H01_Ablate", 23, False, 1, 0.0),
]


def get_balanced_corpus() -> List[str]:
    return [
        "The capital of France is the city of", "The capital of Japan is the bustling city of",
        "The official capital city of Germany is", "The historic capital of Italy is the city of",
        "The capital of Spain is the city of", "The capital of England is the city of",
        "The capital of Russia is the city of", "The capital of Egypt is the city of",
        "The capital of Greece is the city of", "The capital of China is the city of",
        "The capital of Canada is the city of", "The capital of Australia is the city of",
        "Alice handed a red apple to Bob. The apple was received by",
        "David passed the heavy book to Sarah. The book was taken by",
        "John sent a private email to Mary. The email reached",
        "Michael offered a cup of tea to Emma. The tea was accepted by",
        "Tom tossed the baseball to Jack. The ball was caught by",
        "Robert wrote a letter to Lisa. The letter was delivered to",
        "William awarded a shiny medal to Oliver. The medal went to",
        "James threw the yellow frisbee to Lucas. The frisbee was caught by",
        "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by",
        "Henry presented an award to Ethan. The award was received by",
        "Samuel mailed a package to Alexander. The package arrived for",
        "George loaned some money to Noah. The money helped",
        "While Rome has the Colosseum, the Eiffel Tower stands in",
        "Even though Madrid is famous, the capital of Germany is",
        "Berlin has monuments, yet the ancient Colosseum stands in",
        "Although Tokyo is very large, the ancient Parthenon is located in",
        "Despite London being historic, the capital of Spain is definitely",
        "Paris is beautiful, but the capital of Italy is definitely",
        "Egypt has pyramids, but the Eiffel Tower was built in the city of",
        "Although Moscow has the Kremlin, the capital of France is the city of",
    ]


# ==============================================================================
# 3. TRAINING PROBES
# ==============================================================================

def train_learned_probes(harness: DASLiteHarness, prefix_cache: List[Tuple], K: int = 4,
                         scale: float = 6.0, epochs: int = 60, ridge_lambda: float = 1.0):
    d_model = harness.d_model
    V = nn.Parameter(torch.randn(K, d_model, device=harness.device, dtype=torch.float32) * 0.02)
    optimizer = torch.optim.Adam([V], lr=0.05)

    Y_train = []
    for (resid, t1, t2, base_margin, tokens) in prefix_cache:
        row = [harness.ground_truth_delta(tokens, t1, t2, base_margin, layer, is_mlp, head, sc)
               for (_, layer, is_mlp, head, sc) in META_TRAIN_TARGETS]
        Y_train.append(row)
    Y_train = torch.tensor(Y_train, dtype=torch.float32, device=harness.device)

    print(f"\n🔧 Training {K} learned probe directions over {epochs} epochs...")

    for epoch in range(epochs):
        V_normed = V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)
        resp_rows = []
        for (resid, t1, t2, base_margin, tokens) in prefix_cache:
            deltas_k = harness.batched_probe_response(resid, V_normed, scale, t1, t2, base_margin)
            resp_rows.append(deltas_k)

        Resp = torch.stack(resp_rows)  # (N, K)

        I = torch.eye(K, device=harness.device)
        W = torch.linalg.solve(Resp.T @ Resp + ridge_lambda * I, Resp.T @ Y_train)
        pred = Resp @ W
        loss = torch.mean((pred - Y_train) ** 2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"  epoch {epoch:03d} | joint MSE loss = {loss.item():.5f}")

    return (V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)).detach()


# ==============================================================================
# 4. STATISTICAL EVALUATION
# ==============================================================================

def leave_one_out_ridge(X: np.ndarray, y: np.ndarray, alpha: float = 0.5) -> Tuple[float, float]:
    N, D = X.shape
    preds = []
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        preds.append(float(np.dot(X[i], w)))
    preds = np.array(preds)
    mae = float(np.mean(np.abs(preds - y)))
    corr = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
    return mae, corr


def permutation_p_value(X: np.ndarray, y: np.ndarray, mae_observed: float, n_perms: int = 500) -> float:
    shuff_maes = []
    for _ in range(n_perms):
        y_s = np.random.permutation(y)
        m_s, _ = leave_one_out_ridge(X, y_s)
        shuff_maes.append(m_s)
    return float((np.sum(np.array(shuff_maes) <= mae_observed) + 1) / (n_perms + 1))


# ==============================================================================
# 5. MASTER CONTROLLER
# ==============================================================================

def run_milestone_11():
    harness = DASLiteHarness()
    prompts = get_balanced_corpus()

    print("\n" + "=" * 140)
    print("🔬 MILESTONE 11: LEARNED CAUSAL SUBSPACE PILOT ('DAS-LITE')")
    print("=" * 140)

    print("📦 Caching prefix representations...")
    prefix_cache = []
    for p in prompts:
        tokens = harness.model.to_tokens(p)
        resid, t1, t2, base_margin = harness.get_prefix_cache(tokens)
        prefix_cache.append((resid, t1, t2, base_margin, tokens))

    torch.cuda.empty_cache()
    gc.collect()

    V_learned = train_learned_probes(harness, prefix_cache, K=4, scale=6.0, epochs=60)
    torch.cuda.empty_cache()

    print("\n✅ Extracting learned representations...")
    X_learned = []
    with torch.no_grad():
        for (resid, t1, t2, base_margin, tokens) in prefix_cache:
            deltas = harness.batched_probe_response(resid, V_learned, 6.0, t1, t2, base_margin)
            X_learned.append(deltas.cpu().numpy())
    X_learned = np.array(X_learned, dtype=np.float32)

    print("\n" + "=" * 140)
    print("📊 HELD-OUT EVALUATION ON UNSEEN TARGETS")
    print("=" * 140)
    print(f"{'Held-out Target':<18} | {'Learned Phi ρ':<15} | {'Learned MAE':<12} | {'Perm p-value':<14} | {'Verdict'}")
    print("-" * 140)

    results = []
    any_significant = False
    for (name, layer, is_mlp, head, sc) in HELD_OUT_TARGETS:
        y = np.array([harness.ground_truth_delta(tokens, t1, t2, bm, layer, is_mlp, head, sc)
                      for (resid, t1, t2, bm, tokens) in prefix_cache])
        mae, corr = leave_one_out_ridge(X_learned, y)
        p_val = permutation_p_value(X_learned, y, mae, n_perms=500)
        sig = p_val < 0.05
        any_significant = any_significant or sig
        sig_str = "🟢 SIGNIFICANT" if sig else "🔴 NOT SIGNIFICANT"
        print(f"{name:<18} | {corr:+13.4f}   | {mae:10.4f}   | p={p_val:.4f}      | {sig_str}")
        results.append({"target": name, "corr": corr, "mae": mae, "p_value": p_val, "significant": bool(sig)})

    print("-" * 140)
    if any_significant:
        print("👉 PILOT VERDICT: 🟢 LEARNED_REPRESENTATION_SHOWS_ADVANTAGE")
    else:
        print("👉 PILOT VERDICT: 🔴 NO_ADVANTAGE_OVER_FIXED_FINGERPRINT")
    print("=" * 140 + "\n")

    with open("m11_das_lite_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "meta_train_targets": [t[0] for t in META_TRAIN_TARGETS],
            "held_out_results": results,
            "any_significant": bool(any_significant),
        }, f, indent=2)


if __name__ == "__main__":
    run_milestone_11()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_996/1389479613.py:33: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 11: LEARNED CAUSAL SUBSPACE PILOT ('DAS-LITE')
📦 Caching prefix representations...

🔧 Training 4 learned probe directions over 60 epochs...
  epoch 000 | joint MSE loss = 0.16510
  epoch 010 | joint MSE loss = 0.01077
  epoch 020 | joint MSE loss = 0.00823
  epoch 030 | joint MSE loss = 0.00733
  epoch 040 | joint MSE loss = 0.00680
  epoch 050 | joint MSE loss = 0.00612
  epoch 059 | joint MSE loss = 0.00607

✅ Extracting learned representations...

📊 HELD-OUT EVALUATION ON UNSEEN TARGETS
Held-out Target    | Learned Phi ρ   | Learned MAE  | Perm p-value   | Verdict
--------------------------------------------------------------------------------------------------------------------------------------------
L21H02_Ablate      |       +0.0829   |     0.0803   | p=0.3613      | 🔴 NOT SIGNIFICANT
L14H04_Ablate      |       -0.0094   |     0.3505   | p=0.0539      | 🔴 NOT SIGNIFICANT
L10H00_Ablate      |       +0.5

In [3]:
"""
m11_1_strict_generalization_audit.py
=====================================
Milestone 11.1: Strict Prompt-Disjoint & Cross-Intervention Generalization Benchmark.

Key Protocols:
  1. Strict 50/50 Prompt Split: 16 Train Prompts (for learning V) vs. 16 Test Prompts (Held-out).
  2. Strict Target Split: 4 Meta-Train Targets vs. 4 Held-Out Macro Targets.
  3. 4-Way Baseline Comparison on TEST Prompts:
     - Learned DAS-Lite Subspace
     - Fixed Handcrafted Fingerprint
     - Random Subspace Control
     - Passive Residual State
  4. Non-parametric Paired Testing across held-out targets.
"""

import torch
import torch.nn as nn
import numpy as np
import json
import gc
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS
# ==============================================================================

class StrictDASLiteHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.d_model = self.model.cfg.d_model
        self.inject_layer = 21
        self.tail_start_layer = 22

    @torch.no_grad()
    def get_prefix_and_fixed_phi(self, tokens: torch.Tensor) -> Tuple[torch.Tensor, int, int, float, np.ndarray, np.ndarray]:
        _, cache = self.model.run_with_cache(
            tokens, stop_at_layer=self.tail_start_layer,
            names_filter=f"blocks.{self.inject_layer}.hook_resid_post"
        )
        resid = cache[f"blocks.{self.inject_layer}.hook_resid_post"].cpu()
        base_logits = self.model(resid.to(self.device), start_at_layer=self.tail_start_layer)[0, -1, :]
        top_ids = torch.topk(base_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((base_logits[t1] - base_logits[t2]).item())

        # Passive State Vector (First 16 normalized components of resid)
        pass_vec = resid[0, -1, :16].float().numpy()
        pass_vec = pass_vec / (np.linalg.norm(pass_vec) + 1e-8)

        # Fixed Handcrafted Micro-Probes (Phi_fixed)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi_fixed = np.array([d1, d2, d3, d4], dtype=np.float32)
        return resid, t1, t2, base_margin, phi_fixed, pass_vec

    def batched_subspace_response(self, resid: torch.Tensor, V_normed: torch.Tensor, scale: float,
                                  t1: int, t2: int, base_margin: float) -> torch.Tensor:
        K = V_normed.shape[0]
        resid_k = resid.to(self.device).repeat(K, 1, 1).to(torch.bfloat16)
        resid_k[:, -1, :] = resid_k[:, -1, :] + scale * V_normed.to(torch.bfloat16)

        logits = self.model(resid_k, start_at_layer=self.tail_start_layer)[:, -1, :]
        deltas = (logits[:, t1] - logits[:, t2]).to(torch.float32) - base_margin
        return deltas

    @torch.no_grad()
    def ground_truth_delta(self, tokens: torch.Tensor, t1: int, t2: int, base_margin: float,
                           layer: int, is_mlp: bool, head: int = 0, scale: float = 0.0) -> float:
        if is_mlp:
            def hook_fn(act, hook): act[:, -1, :] *= scale; return act
            hook_name = f"blocks.{layer}.hook_mlp_out"
        else:
            def hook_fn(act, hook): act[:, -1, head, :] *= scale; return act
            hook_name = f"blocks.{layer}.attn.hook_z"
        logits = self.model.run_with_hooks(tokens, fwd_hooks=[(hook_name, hook_fn)])[0, -1, :]
        return float((logits[t1] - logits[t2]).item()) - base_margin


# ==============================================================================
# 2. DATASETS & TARGET DEFINITIONS
# ==============================================================================

META_TRAIN_TARGETS = [
    ("L22H05_Ablate", 22, False, 5, 0.0),
    ("L22H00_Ablate", 22, False, 0, 0.0),
    ("L21MLP_Damp",   21, True,  0, 0.5),
    ("L18MLP_Damp",   18, True,  0, 0.5),
]

HELD_OUT_TARGETS = [
    ("L21H02_Ablate", 21, False, 2, 0.0),
    ("L14H04_Ablate", 14, False, 4, 0.0),
    ("L10H00_Ablate", 10, False, 0, 0.0),
    ("L23H01_Ablate", 23, False, 1, 0.0),
]

def get_disjoint_corpus() -> Tuple[List[str], List[str]]:
    train_prompts = [
        "The capital of France is the city of", "The capital of Japan is the bustling city of",
        "The official capital city of Germany is", "The historic capital of Italy is the city of",
        "The capital of Spain is the city of", "The capital of England is the city of",
        "The capital of Russia is the city of", "The capital of Egypt is the city of",
        "Alice handed a red apple to Bob. The apple was received by",
        "David passed the heavy book to Sarah. The book was taken by",
        "John sent a private email to Mary. The email reached",
        "Michael offered a cup of tea to Emma. The tea was accepted by",
        "While Rome has the Colosseum, the Eiffel Tower stands in",
        "Even though Madrid is famous, the capital of Germany is",
        "Berlin has monuments, yet the ancient Colosseum stands in",
        "Although Tokyo is very large, the ancient Parthenon is located in"
    ]
    test_prompts = [
        "The capital of Greece is the city of", "The capital of China is the city of",
        "The capital of Canada is the city of", "The capital of Australia is the city of",
        "Tom tossed the baseball to Jack. The ball was caught by",
        "Robert wrote a letter to Lisa. The letter was delivered to",
        "William awarded a shiny medal to Oliver. The medal went to",
        "James threw the yellow frisbee to Lucas. The frisbee was caught by",
        "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by",
        "Henry presented an award to Ethan. The award was received by",
        "Samuel mailed a package to Alexander. The package arrived for",
        "George loaned some money to Noah. The money helped",
        "Despite London being historic, the capital of Spain is definitely",
        "Paris is beautiful, but the capital of Italy is definitely",
        "Egypt has pyramids, but the Eiffel Tower was built in the city of",
        "Although Moscow has the Kremlin, the capital of France is the city of"
    ]
    return train_prompts, test_prompts


# ==============================================================================
# 3. TRAINING PROBES ON TRAIN SPLIT ONLY
# ==============================================================================

def train_probes_on_train_split(harness: StrictDASLiteHarness, train_cache: List[Tuple],
                                K: int = 4, scale: float = 6.0, epochs: int = 60) -> torch.Tensor:
    d_model = harness.d_model
    V = nn.Parameter(torch.randn(K, d_model, device=harness.device, dtype=torch.float32) * 0.02)
    optimizer = torch.optim.Adam([V], lr=0.05)

    Y_train = []
    for (resid, t1, t2, bm, _, _, tokens) in train_cache:
        row = [harness.ground_truth_delta(tokens, t1, t2, bm, layer, is_mlp, head, sc)
               for (_, layer, is_mlp, head, sc) in META_TRAIN_TARGETS]
        Y_train.append(row)
    Y_train = torch.tensor(Y_train, dtype=torch.float32, device=harness.device)

    for epoch in range(epochs):
        V_normed = V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)
        resp_rows = []
        for (resid, t1, t2, bm, _, _, tokens) in train_cache:
            deltas_k = harness.batched_subspace_response(resid, V_normed, scale, t1, t2, bm)
            resp_rows.append(deltas_k)

        Resp = torch.stack(resp_rows)
        I = torch.eye(K, device=harness.device)
        W = torch.linalg.solve(Resp.T @ Resp + 1.0 * I, Resp.T @ Y_train)
        pred = Resp @ W
        loss = torch.mean((pred - Y_train) ** 2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return (V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)).detach()


# ==============================================================================
# 4. EVALUATION RIGOR
# ==============================================================================

def evaluate_loo_ridge(X: np.ndarray, y: np.ndarray, alpha: float = 0.5) -> Tuple[float, float]:
    N, D = X.shape
    preds = []
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        preds.append(float(np.dot(X[i], w)))
    preds = np.array(preds)
    mae = float(np.mean(np.abs(preds - y)))
    corr = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
    return mae, corr


# ==============================================================================
# 5. MASTER CONTROLLER
# ==============================================================================

def run_milestone_11_1():
    harness = StrictDASLiteHarness()
    train_prompts, test_prompts = get_disjoint_corpus()

    print("\n" + "=" * 145)
    print(f"🔬 MILESTONE 11.1: STRICT PROMPT-DISJOINT & CROSS-INTERVENTION GENERALIZATION AUDIT")
    print(f"   Train Split: {len(train_prompts)} Prompts | Test Split: {len(test_prompts)} Prompts (100% DISJOINT)")
    print("=" * 145)

    # 1. Cache Representations
    print("📦 Caching prefix data for Train and Test splits...")
    train_cache, test_cache = [], []
    for p in train_prompts:
        tok = harness.model.to_tokens(p)
        res, t1, t2, bm, p_fix, p_pass = harness.get_prefix_and_fixed_phi(tok)
        train_cache.append((res, t1, t2, bm, p_fix, p_pass, tok))

    for p in test_prompts:
        tok = harness.model.to_tokens(p)
        res, t1, t2, bm, p_fix, p_pass = harness.get_prefix_and_fixed_phi(tok)
        test_cache.append((res, t1, t2, bm, p_fix, p_pass, tok))

    # 2. Train V on Train Prompts ONLY
    print("🔧 Learning probe directions V on Train Prompts ONLY...")
    V_learned = train_probes_on_train_split(harness, train_cache, K=4, epochs=60)

    # 3. Create Random Subspace Baseline (Same shape and norm)
    V_rand = torch.randn(4, harness.d_model, device=harness.device, dtype=torch.float32)
    V_rand = (V_rand / V_rand.norm(dim=1, keepdim=True)).detach()

    # 4. Extract All 4 Representations on TEST PROMPTS ONLY
    print("✅ Extracting all 4 representations on unseen TEST PROMPTS...")
    X_learned_test, X_fixed_test, X_rand_test, X_pass_test = [], [], [], []

    with torch.no_grad():
        for (res, t1, t2, bm, p_fix, p_pass, _) in test_cache:
            d_learn = harness.batched_subspace_response(res, V_learned, 6.0, t1, t2, bm).cpu().numpy()
            d_rand = harness.batched_subspace_response(res, V_rand, 6.0, t1, t2, bm).cpu().numpy()
            X_learned_test.append(d_learn)
            X_rand_test.append(d_rand)
            X_fixed_test.append(p_fix)
            X_pass_test.append(p_pass)

    X_learned_test = np.array(X_learned_test, dtype=np.float32)
    X_fixed_test = np.array(X_fixed_test, dtype=np.float32)
    X_rand_test = np.array(X_rand_test, dtype=np.float32)
    X_pass_test = np.array(X_pass_test, dtype=np.float32)

    # -------------------------------------------------------------------------
    # 5. BENCHMARK ON HELD-OUT TARGETS (TEST PROMPTS ONLY)
    # -------------------------------------------------------------------------
    print("\n" + "=" * 145)
    print("📊 4-WAY COMPARATIVE BENCHMARK ON UNSEEN TARGETS & UNSEEN PROMPTS")
    print("=" * 145)
    print(f"{'Held-Out Target':<18} | {'Learned DAS ρ':<14} | {'Fixed Phi ρ':<12} | {'Random Sub ρ':<13} | {'Passive ρ':<11} | {'Advantage Status'}")
    print("-" * 145)

    comp_results = []
    for (name, layer, is_mlp, head, sc) in HELD_OUT_TARGETS:
        y_test = np.array([harness.ground_truth_delta(tok, t1, t2, bm, layer, is_mlp, head, sc)
                           for (_, t1, t2, bm, _, _, tok) in test_cache])

        _, corr_learn = evaluate_loo_ridge(X_learned_test, y_test)
        _, corr_fixed = evaluate_loo_ridge(X_fixed_test, y_test)
        _, corr_rand = evaluate_loo_ridge(X_rand_test, y_test)
        _, corr_pass = evaluate_loo_ridge(X_pass_test, y_test)

        is_superior = corr_learn > max(corr_fixed, corr_rand, corr_pass)
        stat_str = "🟢 SUPERIOR" if is_superior else "🔴 NOT SUPERIOR"

        print(f"{name:<18} | {corr_learn:+12.4f}   | {corr_fixed:+10.4f}   | {corr_rand:+11.4f}   | {corr_pass:+9.4f}   | {stat_str}")
        comp_results.append({
            "target": name,
            "corr_learned": corr_learn,
            "corr_fixed": corr_fixed,
            "corr_rand": corr_rand,
            "corr_pass": corr_pass,
            "is_superior": is_superior
        })

    print("-" * 145)
    mean_learn = np.mean([r["corr_learned"] for r in comp_results])
    mean_fixed = np.mean([r["corr_fixed"] for r in comp_results])
    mean_rand = np.mean([r["corr_rand"] for r in comp_results])
    mean_pass = np.mean([r["corr_pass"] for r in comp_results])

    print(f"Mean Correlation (ρ) | Learned: {mean_learn:+.4f} | Fixed: {mean_fixed:+.4f} | Random: {mean_rand:+.4f} | Passive: {mean_pass:+.4f}")
    print("-" * 145)

    if mean_learn > mean_fixed and mean_learn > mean_rand and mean_learn >= 0.30:
        print("👉 BENCHMARK VERDICT: 🟢 ROBUST_GENERALIZABLE_CAUSAL_SUBSPACE_VALIDATED")
        print("   (Learned directions generalize across both unseen prompts AND unseen causal interventions)")
    else:
        print("👉 BENCHMARK VERDICT: 🔴 TRANSDUCTIVE_OVERFITTING_CONFIRMED")
        print("   (Learned representation fails to outperform baselines under strict prompt-disjoint evaluation)")
    print("=" * 145 + "\n")

    with open("m11_1_disjoint_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "results": comp_results,
            "means": {"learned": mean_learn, "fixed": mean_fixed, "rand": mean_rand, "pass": mean_pass}
        }, f, indent=2)


if __name__ == "__main__":
    run_milestone_11_1()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_996/1225560805.py:38: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 11.1: STRICT PROMPT-DISJOINT & CROSS-INTERVENTION GENERALIZATION AUDIT
   Train Split: 16 Prompts | Test Split: 16 Prompts (100% DISJOINT)
📦 Caching prefix data for Train and Test splits...
🔧 Learning probe directions V on Train Prompts ONLY...
✅ Extracting all 4 representations on unseen TEST PROMPTS...

📊 4-WAY COMPARATIVE BENCHMARK ON UNSEEN TARGETS & UNSEEN PROMPTS
Held-Out Target    | Learned DAS ρ  | Fixed Phi ρ  | Random Sub ρ  | Passive ρ   | Advantage Status
-------------------------------------------------------------------------------------------------------------------------------------------------
L21H02_Ablate      |      -0.0004   |    -0.2742   |     +0.2993   |   -0.5482   | 🔴 NOT SUPERIOR
L14H04_Ablate      |      +0.0895   |    -0.1803   |     +0.0404   |   +0.0458   | 🟢 SUPERIOR
L10H00_Ablate      |      +0.3757   |    +0.3898   |     +0.5957   |   -0.3388   | 🔴 NOT SUPERIOR
L23H01_Ablate 

In [ ]:
"""
m11_2_diagnostic_audit.py
==========================
Milestone 11.2: Multi-Seed Random Subspace vs. Dimensionality-Swept DAS Audit.

Protocol:
  1. Strict Prompt Split: 16 Train vs. 16 Test (Disjoint).
  2. Multi-Seed Random Distribution: Evaluates N_SEEDS=30 distinct random subspaces.
  3. Dimensionality Scaling: Tests K in {1, 2, 4, 8} dimensions.
  4. Statistical Verdict: Compares Learned DAS against the true empirical Null Distribution.
"""

import torch
import torch.nn as nn
import numpy as np
import json
import gc
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. HARNESS & DATA STRUCTURES
# ==============================================================================

class DiagnosticHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.d_model = self.model.cfg.d_model
        self.inject_layer = 21
        self.tail_start_layer = 22

    @torch.no_grad()
    def get_prefix_cache(self, tokens: torch.Tensor) -> Tuple[torch.Tensor, int, int, float]:
        _, cache = self.model.run_with_cache(
            tokens, stop_at_layer=self.tail_start_layer,
            names_filter=f"blocks.{self.inject_layer}.hook_resid_post"
        )
        resid = cache[f"blocks.{self.inject_layer}.hook_resid_post"].cpu()
        base_logits = self.model(resid.to(self.device), start_at_layer=self.tail_start_layer)[0, -1, :]
        top_ids = torch.topk(base_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((base_logits[t1] - base_logits[t2]).item())
        return resid, t1, t2, base_margin

    def batched_subspace_response(self, resid: torch.Tensor, V_normed: torch.Tensor, scale: float,
                                  t1: int, t2: int, base_margin: float) -> torch.Tensor:
        K = V_normed.shape[0]
        resid_k = resid.to(self.device).repeat(K, 1, 1).to(torch.bfloat16)
        resid_k[:, -1, :] = resid_k[:, -1, :] + scale * V_normed.to(torch.bfloat16)
        logits = self.model(resid_k, start_at_layer=self.tail_start_layer)[:, -1, :]
        deltas = (logits[:, t1] - logits[:, t2]).to(torch.float32) - base_margin
        return deltas

    @torch.no_grad()
    def ground_truth_delta(self, tokens: torch.Tensor, t1: int, t2: int, base_margin: float,
                           layer: int, is_mlp: bool, head: int = 0, scale: float = 0.0) -> float:
        if is_mlp:
            def hook_fn(act, hook): act[:, -1, :] *= scale; return act
            hook_name = f"blocks.{layer}.hook_mlp_out"
        else:
            def hook_fn(act, hook): act[:, -1, head, :] *= scale; return act
            hook_name = f"blocks.{layer}.attn.hook_z"
        logits = self.model.run_with_hooks(tokens, fwd_hooks=[(hook_name, hook_fn)])[0, -1, :]
        return float((logits[t1] - logits[t2]).item()) - base_margin


# ==============================================================================
# 2. DATASETS & TARGETS
# ==============================================================================

META_TRAIN_TARGETS = [
    ("L22H05_Ablate", 22, False, 5, 0.0),
    ("L22H00_Ablate", 22, False, 0, 0.0),
    ("L21MLP_Damp",   21, True,  0, 0.5),
    ("L18MLP_Damp",   18, True,  0, 0.5),
]

HELD_OUT_TARGETS = [
    ("L21H02_Ablate", 21, False, 2, 0.0),
    ("L14H04_Ablate", 14, False, 4, 0.0),
    ("L10H00_Ablate", 10, False, 0, 0.0),
    ("L23H01_Ablate", 23, False, 1, 0.0),
]

def get_disjoint_corpus() -> Tuple[List[str], List[str]]:
    train_prompts = [
        "The capital of France is the city of", "The capital of Japan is the bustling city of",
        "The official capital city of Germany is", "The historic capital of Italy is the city of",
        "The capital of Spain is the city of", "The capital of England is the city of",
        "The capital of Russia is the city of", "The capital of Egypt is the city of",
        "Alice handed a red apple to Bob. The apple was received by",
        "David passed the heavy book to Sarah. The book was taken by",
        "John sent a private email to Mary. The email reached",
        "Michael offered a cup of tea to Emma. The tea was accepted by",
        "While Rome has the Colosseum, the Eiffel Tower stands in",
        "Even though Madrid is famous, the capital of Germany is",
        "Berlin has monuments, yet the ancient Colosseum stands in",
        "Although Tokyo is very large, the ancient Parthenon is located in"
    ]
    test_prompts = [
        "The capital of Greece is the city of", "The capital of China is the city of",
        "The capital of Canada is the city of", "The capital of Australia is the city of",
        "Tom tossed the baseball to Jack. The ball was caught by",
        "Robert wrote a letter to Lisa. The letter was delivered to",
        "William awarded a shiny medal to Oliver. The medal went to",
        "James threw the yellow frisbee to Lucas. The frisbee was caught by",
        "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by",
        "Henry presented an award to Ethan. The award was received by",
        "Samuel mailed a package to Alexander. The package arrived for",
        "George loaned some money to Noah. The money helped",
        "Despite London being historic, the capital of Spain is definitely",
        "Paris is beautiful, but the capital of Italy is definitely",
        "Egypt has pyramids, but the Eiffel Tower was built in the city of",
        "Although Moscow has the Kremlin, the capital of France is the city of"
    ]
    return train_prompts, test_prompts


# ==============================================================================
# 3. TRAINING & EVALUATION FUNCTIONS
# ==============================================================================

def train_das(harness: DiagnosticHarness, train_cache: List[Tuple], K: int, epochs: int = 60) -> torch.Tensor:
    V = nn.Parameter(torch.randn(K, harness.d_model, device=harness.device, dtype=torch.float32) * 0.02)
    optimizer = torch.optim.Adam([V], lr=0.05)

    Y_train = []
    for (resid, t1, t2, bm, tokens) in train_cache:
        row = [harness.ground_truth_delta(tokens, t1, t2, bm, layer, is_mlp, head, sc)
               for (_, layer, is_mlp, head, sc) in META_TRAIN_TARGETS]
        Y_train.append(row)
    Y_train = torch.tensor(Y_train, dtype=torch.float32, device=harness.device)

    for _ in range(epochs):
        V_normed = V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)
        resp_rows = []
        for (resid, t1, t2, bm, tokens) in train_cache:
            deltas_k = harness.batched_subspace_response(resid, V_normed, 6.0, t1, t2, bm)
            resp_rows.append(deltas_k)

        Resp = torch.stack(resp_rows)
        I = torch.eye(K, device=harness.device)
        W = torch.linalg.solve(Resp.T @ Resp + 1.0 * I, Resp.T @ Y_train)
        pred = Resp @ W
        loss = torch.mean((pred - Y_train) ** 2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return (V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)).detach()


def evaluate_representation(X: np.ndarray, y_all: List[np.ndarray], alpha: float = 0.5) -> float:
    N, D = X.shape
    corrs = []
    for y in y_all:
        preds = []
        for i in range(N):
            tr_idx = [j for j in range(N) if j != i]
            X_tr, y_tr = X[tr_idx], y[tr_idx]
            w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
            preds.append(float(np.dot(X[i], w)))
        preds = np.array(preds)
        r = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
        corrs.append(r)
    return float(np.mean(corrs))


# ==============================================================================
# 4. MASTER CONTROLLER
# ==============================================================================

def run_milestone_11_2():
    harness = DiagnosticHarness()
    train_prompts, test_prompts = get_disjoint_corpus()

    print("\n" + "=" * 135)
    print("🔬 MILESTONE 11.2: MULTI-SEED RANDOM SUBSPACE DISTRIBUTION & DIMENSIONALITY AUDIT")
    print("=" * 135)

    train_cache, test_cache = [], []
    for p in train_prompts:
        tok = harness.model.to_tokens(p)
        res, t1, t2, bm = harness.get_prefix_cache(tok)
        train_cache.append((res, t1, t2, bm, tok))

    for p in test_prompts:
        tok = harness.model.to_tokens(p)
        res, t1, t2, bm = harness.get_prefix_cache(tok)
        test_cache.append((res, t1, t2, bm, tok))

    y_test_all = [
        np.array([harness.ground_truth_delta(tok, t1, t2, bm, layer, is_mlp, head, sc)
                  for (res, t1, t2, bm, tok) in test_cache])
        for (_, layer, is_mlp, head, sc) in HELD_OUT_TARGETS
    ]

    # Test 1: Multi-Seed Random Subspace Distribution at K=4
    N_SEEDS = 30
    print(f"\n📊 1. EVALUATING RANDOM SUBSPACE DISTRIBUTION ACROSS {N_SEEDS} SEEDS (K=4)...")
    random_seed_scores = []

    for s in range(N_SEEDS):
        torch.manual_seed(s + 100)
        V_rand = torch.randn(4, harness.d_model, device=harness.device, dtype=torch.float32)
        V_rand = (V_rand / V_rand.norm(dim=1, keepdim=True)).detach()

        X_rand = []
        with torch.no_grad():
            for (res, t1, t2, bm, _) in test_cache:
                d = harness.batched_subspace_response(res, V_rand, 6.0, t1, t2, bm).cpu().numpy()
                X_rand.append(d)
        X_rand = np.array(X_rand, dtype=np.float32)
        score = evaluate_representation(X_rand, y_test_all)
        random_seed_scores.append(score)

    rand_mean = float(np.mean(random_seed_scores))
    rand_std = float(np.std(random_seed_scores))
    rand_p95 = float(np.percentile(random_seed_scores, 95))
    rand_p05 = float(np.percentile(random_seed_scores, 5))

    print(f"  • Random Subspace Mean ρ (μ) : {rand_mean:+.4f}")
    print(f"  • Standard Deviation (σ)    : {rand_std:.4f}")
    print(f"  • 90% Confidence Interval   : [{rand_p05:+.4f}, {rand_p95:+.4f}]")

    # Test 2: Dimensionality Sweep K in {1, 2, 4, 8}
    print("\n📊 2. DIMENSIONALITY SWEEP: LEARNED DAS VS. RANDOM SUBSPACE")
    print("-" * 135)
    print(f"{'Dimension (K)':<15} | {'Learned DAS Mean ρ':<22} | {'Random Mean ρ (30 Seeds)':<26} | {'Gap (DAS - Rand)'}")
    print("-" * 135)

    sweep_results = {}
    for K in [1, 2, 4, 8]:
        torch.manual_seed(SEED)
        V_das = train_das(harness, train_cache, K=K, epochs=60)

        X_das = []
        with torch.no_grad():
            for (res, t1, t2, bm, _) in test_cache:
                d = harness.batched_subspace_response(res, V_das, 6.0, t1, t2, bm).cpu().numpy()
                X_das.append(d)
        X_das = np.array(X_das, dtype=np.float32)
        das_score = evaluate_representation(X_das, y_test_all)

        # Average over 10 random seeds for this specific dimension
        dim_rand_scores = []
        for s in range(10):
            torch.manual_seed(s + 200)
            V_r = torch.randn(K, harness.d_model, device=harness.device, dtype=torch.float32)
            V_r = (V_r / V_r.norm(dim=1, keepdim=True)).detach()
            X_r = []
            with torch.no_grad():
                for (res, t1, t2, bm, _) in test_cache:
                    d_r = harness.batched_subspace_response(res, V_r, 6.0, t1, t2, bm).cpu().numpy()
                    X_r.append(d_r)
            dim_rand_scores.append(evaluate_representation(np.array(X_r, dtype=np.float32), y_test_all))

        k_rand_mean = float(np.mean(dim_rand_scores))
        gap = das_score - k_rand_mean
        print(f"K = {K:<11} | {das_score:+20.4f}   | {k_rand_mean:+24.4f}   | {gap:+16.4f}")
        sweep_results[f"K_{K}"] = {"das": das_score, "random_mean": k_rand_mean, "gap": gap}

    print("-" * 135)
    if rand_mean >= 0.25 and rand_std <= 0.10:
        print("👉 DIAGNOSTIC VERDICT: 🟢 INTRINSIC_GEOMETRY_PRESERVATION_CONFIRMED")
        print("   (Random projection is consistently robust across seeds; high baseline is an intrinsic property of the residual stream)")
    elif rand_mean < 0.15:
        print("👉 DIAGNOSTIC VERDICT: 🟡 M11_1_WAS_A_RANDOM_FLUKE")
    else:
        print("👉 DIAGNOSTIC VERDICT: 🔴 OPTIMIZATION_OBJECTIVE_MISMATCH")
    print("=" * 135 + "\n")

    with open("m11_2_diagnostic_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "random_seed_distribution_k4": {
                "mean": rand_mean, "std": rand_std, "p95": rand_p95, "p05": rand_p05, "scores": random_seed_scores
            },
            "dimensionality_sweep": sweep_results
        }, f, indent=2)


if __name__ == "__main__":
    run_milestone_11_2()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_996/480642095.py:34: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 11.2: MULTI-SEED RANDOM SUBSPACE DISTRIBUTION & DIMENSIONALITY AUDIT

📊 1. EVALUATING RANDOM SUBSPACE DISTRIBUTION ACROSS 30 SEEDS (K=4)...
  • Random Subspace Mean ρ (μ) : +0.0760
  • Standard Deviation (σ)    : 0.1383
  • 90% Confidence Interval   : [-0.1399, +0.3337]

📊 2. DIMENSIONALITY SWEEP: LEARNED DAS VS. RANDOM SUBSPACE
---------------------------------------------------------------------------------------------------------------------------------------
Dimension (K)   | Learned DAS Mean ρ     | Random Mean ρ (30 Seeds)   | Gap (DAS - Rand)
---------------------------------------------------------------------------------------------------------------------------------------
K = 1           |              -0.2489   |                  -0.1055   |          -0.1434
K = 2           |              +0.0670   |                  +0.0451   |          +0.0219
K = 4           |              +0.1103   |          

In [ ]:
"""
m11_3_distributional_causal_audit.py
=====================================
Milestone 11.3: Multi-Seed Distributional Benchmark (DAS vs. Random Subspaces).

Evaluates whether the distribution of learned DAS representations across 20 independent
initializations systematically dominates the empirical null distribution (100 random subspaces)
under strict prompt-disjoint evaluation for K in {4, 8}.
"""

import torch
import torch.nn as nn
import numpy as np
import json
import gc
from typing import Dict, List, Tuple
from scipy.stats import mannwhitneyu
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS
# ==============================================================================

class FastDistributionalHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.d_model = self.model.cfg.d_model
        self.inject_layer = 21
        self.tail_start_layer = 22

    @torch.no_grad()
    def get_prefix_cache(self, tokens: torch.Tensor) -> Tuple[torch.Tensor, int, int, float]:
        _, cache = self.model.run_with_cache(
            tokens, stop_at_layer=self.tail_start_layer,
            names_filter=f"blocks.{self.inject_layer}.hook_resid_post"
        )
        resid = cache[f"blocks.{self.inject_layer}.hook_resid_post"].cpu()
        base_logits = self.model(resid.to(self.device), start_at_layer=self.tail_start_layer)[0, -1, :]
        top_ids = torch.topk(base_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((base_logits[t1] - base_logits[t2]).item())
        return resid, t1, t2, base_margin

    def batched_subspace_response(self, resid: torch.Tensor, V_normed: torch.Tensor, scale: float,
                                  t1: int, t2: int, base_margin: float) -> torch.Tensor:
        K = V_normed.shape[0]
        resid_k = resid.to(self.device).repeat(K, 1, 1).to(torch.bfloat16)
        resid_k[:, -1, :] = resid_k[:, -1, :] + scale * V_normed.to(torch.bfloat16)
        logits = self.model(resid_k, start_at_layer=self.tail_start_layer)[:, -1, :]
        deltas = (logits[:, t1] - logits[:, t2]).to(torch.float32) - base_margin
        return deltas

    @torch.no_grad()
    def ground_truth_delta(self, tokens: torch.Tensor, t1: int, t2: int, base_margin: float,
                           layer: int, is_mlp: bool, head: int = 0, scale: float = 0.0) -> float:
        if is_mlp:
            def hook_fn(act, hook): act[:, -1, :] *= scale; return act
            hook_name = f"blocks.{layer}.hook_mlp_out"
        else:
            def hook_fn(act, hook): act[:, -1, head, :] *= scale; return act
            hook_name = f"blocks.{layer}.attn.hook_z"
        logits = self.model.run_with_hooks(tokens, fwd_hooks=[(hook_name, hook_fn)])[0, -1, :]
        return float((logits[t1] - logits[t2]).item()) - base_margin


# ==============================================================================
# 2. CORPUS & TARGET DEFINITIONS
# ==============================================================================

META_TRAIN_TARGETS = [
    ("L22H05_Ablate", 22, False, 5, 0.0),
    ("L22H00_Ablate", 22, False, 0, 0.0),
    ("L21MLP_Damp",   21, True,  0, 0.5),
    ("L18MLP_Damp",   18, True,  0, 0.5),
]

HELD_OUT_TARGETS = [
    ("L21H02_Ablate", 21, False, 2, 0.0),
    ("L14H04_Ablate", 14, False, 4, 0.0),
    ("L10H00_Ablate", 10, False, 0, 0.0),
    ("L23H01_Ablate", 23, False, 1, 0.0),
]

def get_disjoint_corpus() -> Tuple[List[str], List[str]]:
    train_prompts = [
        "The capital of France is the city of", "The capital of Japan is the bustling city of",
        "The official capital city of Germany is", "The historic capital of Italy is the city of",
        "The capital of Spain is the city of", "The capital of England is the city of",
        "The capital of Russia is the city of", "The capital of Egypt is the city of",
        "Alice handed a red apple to Bob. The apple was received by",
        "David passed the heavy book to Sarah. The book was taken by",
        "John sent a private email to Mary. The email reached",
        "Michael offered a cup of tea to Emma. The tea was accepted by",
        "While Rome has the Colosseum, the Eiffel Tower stands in",
        "Even though Madrid is famous, the capital of Germany is",
        "Berlin has monuments, yet the ancient Colosseum stands in",
        "Although Tokyo is very large, the ancient Parthenon is located in"
    ]
    test_prompts = [
        "The capital of Greece is the city of", "The capital of China is the city of",
        "The capital of Canada is the city of", "The capital of Australia is the city of",
        "Tom tossed the baseball to Jack. The ball was caught by",
        "Robert wrote a letter to Lisa. The letter was delivered to",
        "William awarded a shiny medal to Oliver. The medal went to",
        "James threw the yellow frisbee to Lucas. The frisbee was caught by",
        "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by",
        "Henry presented an award to Ethan. The award was received by",
        "Samuel mailed a package to Alexander. The package arrived for",
        "George loaned some money to Noah. The money helped",
        "Despite London being historic, the capital of Spain is definitely",
        "Paris is beautiful, but the capital of Italy is definitely",
        "Egypt has pyramids, but the Eiffel Tower was built in the city of",
        "Although Moscow has the Kremlin, the capital of France is the city of"
    ]
    return train_prompts, test_prompts


# ==============================================================================
# 3. STATISTICAL EVALUATION FUNCTIONS
# ==============================================================================

def train_das_instance(harness: FastDistributionalHarness, train_cache: List[Tuple],
                       Y_train: torch.Tensor, K: int, seed: int, epochs: int = 50) -> torch.Tensor:
    torch.manual_seed(seed)
    V = nn.Parameter(torch.randn(K, harness.d_model, device=harness.device, dtype=torch.float32) * 0.02)
    optimizer = torch.optim.Adam([V], lr=0.05)

    for _ in range(epochs):
        V_normed = V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)
        resp_rows = []
        for (resid, t1, t2, bm, tokens) in train_cache:
            deltas_k = harness.batched_subspace_response(resid, V_normed, 6.0, t1, t2, bm)
            resp_rows.append(deltas_k)

        Resp = torch.stack(resp_rows)
        I = torch.eye(K, device=harness.device)
        W = torch.linalg.solve(Resp.T @ Resp + 1.0 * I, Resp.T @ Y_train)
        pred = Resp @ W
        loss = torch.mean((pred - Y_train) ** 2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return (V / V.norm(dim=1, keepdim=True).clamp_min(1e-6)).detach()


def evaluate_representation_mean_corr(X: np.ndarray, y_all: List[np.ndarray], alpha: float = 0.5) -> float:
    N, D = X.shape
    corrs = []
    for y in y_all:
        preds = []
        for i in range(N):
            tr_idx = [j for j in range(N) if j != i]
            X_tr, y_tr = X[tr_idx], y[tr_idx]
            w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
            preds.append(float(np.dot(X[i], w)))
        preds = np.array(preds)
        r = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
        corrs.append(r)
    return float(np.mean(corrs))


# ==============================================================================
# 4. MASTER CONTROLLER
# ==============================================================================

def run_milestone_11_3():
    harness = FastDistributionalHarness()
    train_prompts, test_prompts = get_disjoint_corpus()

    print("\n" + "=" * 145)
    print(f"🔬 MILESTONE 11.3: RIGOROUS MULTI-SEED DISTRIBUTIONAL AUDIT (K=4 vs. K=8)")
    print(f"   20 Trained DAS Initializations vs. 100 Random Subspaces (100% Prompt-Disjoint)")
    print("=" * 145)

    # 1. Cache Representations
    train_cache, test_cache = [], []
    for p in train_prompts:
        tok = harness.model.to_tokens(p)
        res, t1, t2, bm = harness.get_prefix_cache(tok)
        train_cache.append((res, t1, t2, bm, tok))

    for p in test_prompts:
        tok = harness.model.to_tokens(p)
        res, t1, t2, bm = harness.get_prefix_cache(tok)
        test_cache.append((res, t1, t2, bm, tok))

    # 2. Extract Ground-Truth Meta-Train and Held-Out Targets
    Y_train_list = []
    for (resid, t1, t2, bm, tokens) in train_cache:
        row = [harness.ground_truth_delta(tokens, t1, t2, bm, layer, is_mlp, head, sc)
               for (_, layer, is_mlp, head, sc) in META_TRAIN_TARGETS]
        Y_train_list.append(row)
    Y_train = torch.tensor(Y_train_list, dtype=torch.float32, device=harness.device)

    y_test_heldout = [
        np.array([harness.ground_truth_delta(tok, t1, t2, bm, layer, is_mlp, head, sc)
                  for (res, t1, t2, bm, tok) in test_cache])
        for (_, layer, is_mlp, head, sc) in HELD_OUT_TARGETS
    ]

    N_DAS_SEEDS = 20
    N_RAND_SEEDS = 100
    benchmark_data = {}

    for K in [4, 8]:
        print(f"\n⚙️ RUNNING EXPERIMENT FOR DIMENSION K = {K}...")

        # A. Train N_DAS_SEEDS DAS instances
        print(f"  • Training {N_DAS_SEEDS} independent DAS instances across different initialization seeds...")
        das_correlations = []
        for s in range(N_DAS_SEEDS):
            V_learned = train_das_instance(harness, train_cache, Y_train, K=K, seed=s * 17 + 42, epochs=50)
            X_das = []
            with torch.no_grad():
                for (res, t1, t2, bm, _) in test_cache:
                    d = harness.batched_subspace_response(res, V_learned, 6.0, t1, t2, bm).cpu().numpy()
                    X_das.append(d)
            das_score = evaluate_representation_mean_corr(np.array(X_das, dtype=np.float32), y_test_heldout)
            das_correlations.append(das_score)

        # B. Sample N_RAND_SEEDS Random Subspaces
        print(f"  • Sampling {N_RAND_SEEDS} isotropic random subspaces...")
        rand_correlations = []
        for s in range(N_RAND_SEEDS):
            torch.manual_seed(s * 31 + 1000)
            V_rand = torch.randn(K, harness.d_model, device=harness.device, dtype=torch.float32)
            V_rand = (V_rand / V_rand.norm(dim=1, keepdim=True)).detach()
            X_r = []
            with torch.no_grad():
                for (res, t1, t2, bm, _) in test_cache:
                    d = harness.batched_subspace_response(res, V_rand, 6.0, t1, t2, bm).cpu().numpy()
                    X_r.append(d)
            r_score = evaluate_representation_mean_corr(np.array(X_r, dtype=np.float32), y_test_heldout)
            rand_correlations.append(r_score)

        das_arr = np.array(das_correlations)
        rand_arr = np.array(rand_correlations)

        # C. Non-parametric Statistical Comparison
        u_stat, p_val = mannwhitneyu(das_arr, rand_arr, alternative='greater')
        prob_superior = float(np.mean(das_arr[:, None] > rand_arr[None, :]))
        cohen_d = (np.mean(das_arr) - np.mean(rand_arr)) / (np.sqrt((np.var(das_arr) + np.var(rand_arr)) / 2.0) + 1e-8)

        benchmark_data[f"K_{K}"] = {
            "das_mean": float(np.mean(das_arr)),
            "das_std": float(np.std(das_arr)),
            "das_median": float(np.median(das_arr)),
            "das_ci95": [float(np.percentile(das_arr, 2.5)), float(np.percentile(das_arr, 97.5))],
            "rand_mean": float(np.mean(rand_arr)),
            "rand_std": float(np.std(rand_arr)),
            "rand_median": float(np.median(rand_arr)),
            "rand_ci95": [float(np.percentile(rand_arr, 2.5)), float(np.percentile(rand_arr, 97.5))],
            "p_superior": prob_superior,
            "mann_whitney_u": float(u_stat),
            "mann_whitney_p": float(p_val),
            "cohen_d": float(cohen_d),
        }

    # -------------------------------------------------------------------------
    # PRINT RESULTS SUMMARY
    # -------------------------------------------------------------------------
    print("\n" + "=" * 145)
    print("📊 DISTRIBUTIONAL COMPARISON TABLE (Test Prompts & Held-out Targets)")
    print("=" * 145)
    print(f"{'Dim (K)':<8} | {'DAS Mean (±σ)':<20} | {'DAS Median [95% CI]':<26} | {'Rand Mean (±σ)':<20} | {'P(DAS > Rand)':<15} | {'Mann-Whitney p':<16} | {'Cohen d'}")
    print("-" * 145)

    for K in [4, 8]:
        d = benchmark_data[f"K_{K}"]
        das_m_str = f"{d['das_mean']:+.4f} (±{d['das_std']:.3f})"
        das_ci_str = f"{d['das_median']:+.4f} [{d['das_ci95'][0]:+.3f}, {d['das_ci95'][1]:+.3f}]"
        rand_m_str = f"{d['rand_mean']:+.4f} (±{d['rand_std']:.3f})"

        print(f"K = {K:<4} | {das_m_str:<20} | {das_ci_str:<26} | {rand_m_str:<20} | {d['p_superior']*100:6.1f}%         | p = {d['mann_whitney_p']:.4e}     | {d['cohen_d']:+6.2f}")

    print("-" * 145)
    k8_res = benchmark_data["K_8"]
    if k8_res["mann_whitney_p"] < 0.01 and k8_res["p_superior"] >= 0.65:
        print("👉 BENCHMARK VERDICT: 🟢 SYSTEMATIC_CAUSAL_SUBSPACE_ADVANTAGE_CONFIRMED")
        print("   (Learned representation distribution significantly dominates random subspaces at K=8)")
    elif k8_res["mann_whitney_p"] < 0.05:
        print("👉 BENCHMARK VERDICT: 🟡 WEAK_OR_BORDERLINE_SUBSPACE_ADVANTAGE")
    else:
        print("👉 BENCHMARK VERDICT: 🔴 NO_SYSTEMATIC_ADVANTAGE_OVER_RANDOM_GEOMETRY")
    print("=" * 145 + "\n")

    with open("m11_3_distributional_results.json", "w", encoding="utf-8") as f:
        json.dump(benchmark_data, f, indent=2)


if __name__ == "__main__":
    run_milestone_11_3()